# Sentiment Analysis on Twitter Data using Apache Spark

**Dataset:** Sentiment140 (1.6M labelled tweets) · **Engine:** PySpark / Spark MLlib

This notebook was the original Databricks analysis, reconstructed faithfully from the
exported `docs/Sentiment140_Spark.html`. It is written for a Databricks workspace where
`spark` is pre-defined and the CSV lives on DBFS. To run it **outside** Databricks, either
add a `SparkSession` at the top, or use the standalone, parameterized
[`src/sentiment_pipeline.py`](../src/sentiment_pipeline.py) which does that for you.


**Project Title:** Sentiment Analysis on Twitter Data using Apache Spark (Sentiment140 Dataset)
---

**1. Introduction and Overview**

People share their thoughts and emotions on social media platforms like Twitter every day. These short texts are often filled with opinions, reactions, or feelings about products, events, and experiences. Being able to automatically understand the tone or sentiment behind such messages is valuable for businesses, researchers, and developers alike. This task is known as sentiment analysis — the process of determining whether a piece of text expresses a positive or negative emotion.

This project focuses on classifying tweets into two categories: positive and negative. We use the Sentiment140 dataset, which includes 1.6 million tweets already labeled with sentiment. This makes it possible to train machine learning models to recognize patterns and make predictions on new tweets.

Because the dataset is large, we use Apache Spark — a powerful big data processing framework — to handle everything efficiently. The entire workflow is implemented on Databricks, a platform that supports working with Spark in the cloud.

We start by cleaning the tweet text, turning it into numbers using a technique called TF-IDF, and then train two models: Logistic Regression and Naive Bayes. We measure how well these models work using common metrics like accuracy and F1 score.

This report explains each step clearly, from preparing the data to evaluating the models, showing how Spark can help with real-world language problems.

---

**2. Task Definition**

The main goal of this project is to understand what people feel based on what they write on Twitter. We take a large number of tweets and try to figure out if the person who wrote each one is feeling positive (happy, excited) or negative (sad, frustrated).

To do this, we use machine learning. We first clean the tweet text to remove things like links and special characters. Then we turn the words into numbers using a method called TF-IDF, which helps the computer understand which words are important.

After that, we train two models: one using Logistic Regression and the other using Naive Bayes. These models learn from a part of the tweets, and then we test how well they can predict the sentiment of tweets they haven't seen before.

The task is to accurately classify each tweet into one of two categories:

* Positive (label 1)
* Negative (label 0)

By comparing the results of the two models, we can see which one does a better job at this classification task.


In [ ]:
# --------------------------------------------
# 🔧 LIBRARIES FOR SPARK NLP & ML
# --------------------------------------------

# PySpark SQL and UDFs
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, regexp_replace, split, trim, when, count, desc, to_timestamp
from pyspark.sql.types import StringType

# MLlib: Features, Classifiers, Evaluation
from pyspark.ml.feature import HashingTF, IDF, CountVectorizer, StringIndexer
from pyspark.ml.classification import LogisticRegression, NaiveBayes
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline

# Text Preprocessing Helpers
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Optional: Pandas and Plotting
import pandas as pd
import matplotlib.pyplot as plt

# NLTK Downloads
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

**3. Data Description**

At the beginning of the analysis, we renamed the columns of the dataset to more meaningful names to make the data easier to understand and work with. This step helped us access and reference the data more clearly throughout the project.

We used the Sentiment140 dataset, which contains 1.6 million tweets. Each tweet is labeled to show whether the sentiment is positive or negative. The dataset includes the following information:

**target**: Sentiment label (0 = negative, 4 = positive)

**id**: Unique tweet ID

**date**: The date the tweet was posted

**query**: The search term used (can be blank)

**user**: The username of the person who tweeted

**text**: The full tweet text

In [ ]:
# --------------------------------------------
# 📂 LOAD SENTIMENT140 DATA (CSV)
# --------------------------------------------

# DBFS path where the file was uploaded (via CLI or UI)
file_path = "dbfs:/FileStore/tables/training_1600000_processed_noemoticon-1.csv"

# Load raw CSV with 6 columns (no header)
df = (
    spark.read.format("csv")
         .option("header", "false")
         .option("inferSchema", "false")
         .option("sep", ",")
         .load(file_path)
)

# Rename columns according to Sentiment140 format
df = df.toDF("target", "id", "date", "query", "user", "text")

# Cache for fast access later
df.cache()

# Show schema and preview
df.printSchema()
df.select("target", "text").show(5, truncate=False)

For our task, we only used the target and text columns. All other columns were kept but not used for training. We noticed that the dataset was perfectly balanced, with:

800,000 tweets labeled as negative (0)

800,000 tweets labeled as positive (4)

There were no tweets with empty or null values in the text and target field . 

In [ ]:
# --------------------------------------------
# 🔍 STEP 3: INITIAL DATA EXPLORATION
# --------------------------------------------

# Count total records
total_count = df.count()
print(f"Total tweets: {total_count:,}")

# Check for nulls in key columns
df.select([count(when(col(c).isNull(), c)).alias(c) for c in ['target', 'text']]).show()

# Check class balance: how many positive vs. negative?
df.groupBy("target").count().orderBy("target").show()

In [ ]:

import matplotlib.pyplot as plt
# Convert to Pandas for plotting (safe with 2 rows)
class_counts = df.groupBy("target").count().orderBy("target").toPandas()

# Plot
class_counts.plot(kind="bar", x="target", y="count", legend=False, figsize=(5, 3), title="Sentiment Class Balance")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# --------------------------------------------
# 🔍 CHECK FOR MISSING VALUES
# --------------------------------------------

from pyspark.sql.functions import count, isnan

# Count nulls and empty strings in key columns
df.select([
    count(when(col(c).isNull(), c)).alias(f"{c}_nulls")
    for c in ['target', 'text']
]).show()

df.select([
    count(when(trim(col(c)) == "", c)).alias(f"{c}_empty")
    for c in ['target', 'text']
]).show()

**4. Text Cleaning and Preprocessing**

Text preprocessing is a crucial step in preparing the text for machine learning models. It ensures that the input is clean, consistent, and meaningful. In our project, we included several core preprocessing techniques, and we also expanded it to include additional steps often used in natural language processing tasks. It included both traditional and expanded steps suitable for large-scale sentiment analysis. Below is the exact transformation pipeline used in the project:

Before training any models, we cleaned the tweet text to remove noisy and irrelevant information that could affect the learning process. This step is crucial in Natural Language Processing (NLP) because real-world text often includes unstructured elements such as symbols, links, and emojis.

We applied the following preprocessing steps:

- **Lowercasing**: All tweet text was converted to lowercase to avoid treating the same words in different cases as separate tokens.
- **URL and Mention Removal**: We removed links (e.g., http://...) and Twitter mentions (@username), which do not carry sentiment value.
- **Hashtag and Punctuation Removal**: Hashtags and punctuation were stripped using regular expressions.
- **Whitespace Trimming**: Extra and irregular spaces were replaced with single spaces.
- **Tokenization**: Text was split into words using whitespace-based tokenization.
- **Stopword Removal**: Common stopwords (e.g., 'and', 'the', 'is') were removed using PySpark’s StopWordsRemover to reduce noise.
- **Vectorization (TF-IDF)**: We used Term Frequency-Inverse Document Frequency to transform the tokenized text into numeric features.

These steps were implemented directly in the Spark pipeline, ensuring they were scalable and compatible with the Databricks environment.

This preprocessing step helped ensure that the input to the models was clean, consistent, and in a format suitable for numerical analysis.



In [ ]:
# --------------------------------------------
# 🧹 STEP 4: TEXT CLEANING & preprocessing 
# --------------------------------------------

# Remove URLs, @mentions, hashtags, and punctuation → then lowercase
df_clean = (
    df.select("target", "text")
      .withColumn(
          "text_clean",
          lower(
              regexp_replace(
                  col("text"),
                  r"http\S+|@\S+|#[A-Za-z0-9_]+|[^a-zA-Z\s]",
                  " "
              )
          )
      )
      .withColumn("text_clean", regexp_replace(col("text_clean"), r"\s+", " "))  # remove extra spaces
      .na.drop()
)

# Show before/after
df_clean.select("text", "text_clean").show(5, truncate=False)

In [ ]:
# --------------------------------------------
#        TOKENIZE + TF-IDF VECTORIZE + removing stopwords 
# --------------------------------------------

from pyspark.ml.feature import StopWordsRemover, HashingTF, IDF
from pyspark.sql.functions import split, trim

# Tokenize manually (safe in Community Edition)
df_tokens = df_clean.withColumn("words", split(trim(col("text_clean")), " +"))

# Remove stopwords
df_filtered = StopWordsRemover(inputCol="words", outputCol="filtered_words").transform(df_tokens)

# Convert to TF vector (100K features)
tf = HashingTF(inputCol="words", outputCol="rawFeatures", numFeatures=100_000)
df_tf = tf.transform(df_tokens)

# Apply IDF to scale weights
idf = IDF(inputCol="rawFeatures", outputCol="features")
idf_model = idf.fit(df_tf)
df_tfidf = idf_model.transform(df_tf)

# Preview TF-IDF vector output
df_tfidf.select("text_clean", "features").show(3, truncate=False)

**4. Analysis of the Classification Process**

After cleaning the tweets and transforming the text into numerical features using TF-IDF, we trained and evaluated two different classifiers: Logistic Regression and Naive Bayes.

We split the dataset into training and testing sets with an 80/20 ratio:

Training set size: 1,280,209 tweets

Testing set size: 319,791 tweets

In [ ]:
# --------------------------------------------
# 🤖 STEP 6: ENCODE LABELS + LogisticRegression
# --------------------------------------------

from pyspark.ml.classification import LogisticRegression
from pyspark.sql.functions import when

# Convert sentiment: 0 = negative → 0, 4 = positive → 1
df_final = df_tfidf.withColumn("label", when(col("target") == 4, 1).otherwise(0))

# Split into train and test (80/20)
train_data, test_data = df_final.randomSplit([0.8, 0.2], seed=42)

print(f"Training rows: {train_data.count():,}")
print(f"Testing rows : {test_data.count():,}")

# Train Logistic Regression
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=20)
model = lr.fit(train_data)

print(" Logistic Regression model trained.")

In [ ]:
# --------------------------------------------
# 📊 STEP 7: EVALUATE LogisticRegression PERFORMANCE
# --------------------------------------------

from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Predict on test set
predictions = model.transform(test_data)

# Show sample predictions
predictions.select("text_clean", "label", "prediction", "probability").show(10, truncate=False)

# Evaluators
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_f1  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
evaluator_pre = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
evaluator_rec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")

# Scores
accuracy  = evaluator_acc.evaluate(predictions)
f1_score  = evaluator_f1.evaluate(predictions)
precision = evaluator_pre.evaluate(predictions)
recall    = evaluator_rec.evaluate(predictions)

# Print results
print(f" Accuracy : {accuracy:.4f}")
print(f" F1 Score : {f1_score:.4f}")
print(f" Precision: {precision:.4f}")
print(f" Recall   : {recall:.4f}")

In [ ]:
# Build confusion matrix from predictions of LogisticRegression
conf_matrix = (
    predictions.groupBy("label", "prediction").count()
               .orderBy("label", "prediction")
               .toPandas()
    .pivot(index="label", columns="prediction", values="count")
    .fillna(0)
)

# Plot confusion matrix
import seaborn as sns

plt.figure(figsize=(4, 3))
sns.heatmap(conf_matrix, annot=True, fmt=".0f", cmap="Blues", cbar=False)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

In [ ]:
# --------------------------------------------
# 🤖 STEP 8: NAIVE BAYES CLASSIFIER & EVALUATION   
# --------------------------------------------

from pyspark.ml.classification import NaiveBayes

# Train Naive Bayes model on same training set
nb = NaiveBayes(featuresCol="features", labelCol="label", modelType="multinomial")
nb_model = nb.fit(train_data)

# Predict on test set
nb_predictions = nb_model.transform(test_data)

# Evaluators
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_f1  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
evaluator_pre = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
evaluator_rec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")

# Scores
nb_accuracy  = evaluator_acc.evaluate(nb_predictions)
nb_f1_score  = evaluator_f1.evaluate(nb_predictions)
nb_precision = evaluator_pre.evaluate(nb_predictions)
nb_recall    = evaluator_rec.evaluate(nb_predictions)

# Print results
print(" Naive Bayes Performance:")
print(f" Accuracy : {nb_accuracy:.4f}")
print(f" F1 Score : {nb_f1_score:.4f}")
print(f" Precision: {nb_precision:.4f}")
print(f" Recall   : {nb_recall:.4f}")

In [ ]:
# Generate confusion matrix from Naive Bayes predictions
nb_conf_matrix = (
    nb_predictions.groupBy("label", "prediction")
    .count()
    .orderBy("label", "prediction")
    .toPandas()
    .pivot(index="label", columns="prediction", values="count")
    .fillna(0)
)

# Display the matrix using seaborn
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(4, 3))
sns.heatmap(nb_conf_matrix, annot=True, fmt="g", cmap="Blues", cbar=False)
plt.title("Naive Bayes Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

**5. Model Comparison**

To get a clearer picture of how both models performed, we compared their results side-by-side across four main metrics: accuracy, F1 score, precision, and recall. This helped us understand not just which model was more accurate, but also which one handled class balance better.

From this comparison, we can see that **Logistic Regression** slightly outperforms Naive Bayes in every metric. However, Naive Bayes still produces solid results and is much faster to train, which can be helpful when speed or simplicity is a priority.

A bar chart was also created to visualize this comparison and make the differences easier to spot.

In [ ]:
import matplotlib.pyplot as plt

# Metrics from your project
models = ['Logistic Regression', 'Naive Bayes']
accuracy = [0.7796, 0.7637]
f1 = [0.7795, 0.7637]
precision = [0.7797, 0.7638]
recall = [0.7796, 0.7637]

# Plot grouped bar chart
x = range(len(models))
width = 0.2

plt.figure(figsize=(8, 4))
plt.bar([p - 1.5 * width for p in x], accuracy, width=width, label='Accuracy')
plt.bar([p - 0.5 * width for p in x], f1, width=width, label='F1 Score')
plt.bar([p + 0.5 * width for p in x], precision, width=width, label='Precision')
plt.bar([p + 1.5 * width for p in x], recall, width=width, label='Recall')

plt.xticks(ticks=x, labels=models)
plt.ylim(0.75, 0.79)
plt.ylabel("Score")
plt.title("Model Performance Comparison")
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

**6. Conclusion**

In this project, we successfully built a working sentiment analysis system using Apache Spark and the Sentiment140 dataset. We went through the full pipeline: loading the data, cleaning the tweets, turning the text into numbers, training models, and evaluating how well they performed.

We tested two different models: Logistic Regression and Naive Bayes. Logistic Regression gave us the best results across all the key metrics — accuracy, precision, recall, and F1 score — while Naive Bayes was a bit faster and simpler to run. This shows that even basic machine learning models can give good results if the data is clean and well-prepared.

By doing everything in Spark and using Databricks, we were able to process a large dataset smoothly. This approach can easily scale to more data or more complex models if needed.

Overall, this project shows that it’s possible to analyze millions of tweets and figure out what people are feeling — all with just some thoughtful preprocessing, simple models, and the power of distributed computing.

In [ ]:
# --------------------------------------------
# 💾 SAVE MODEL TO DBFS
# --------------------------------------------

model_path = "dbfs:/FileStore/models/sentiment140_lr_model"
model.write().overwrite().save(model_path)

print(f"Model saved to {model_path}")